In [1]:
import pandas as pd
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, binarize_qrels, load_runs_from_path
import ir_measures
from ir_measures.util import QrelsConverter
from src.data import parse_file_names

from topic_gen import logger
logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "gen_qrels"

predictions = []
names = []
for result_file in os.listdir(BASE_DIR):
    predictions.append(binarize_qrels(
        ir_measures.read_trec_qrels(os.path.join(BASE_DIR, result_file))))
    names.append(result_file)

In [3]:
runs = load_runs_from_path(DATA_DIR_RAW / "trec-2004-runs")

In [ ]:
from ir_measures import nDCG, MAP, RBP

# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[ROSKendallTau(runs, measures=[nDCG@10, MAP@100])],
    # bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 1 / 2950
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 7 / 2944
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 18 / 2933
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:206) Qrels in reference but not in predictions: 0 / 2951


In [3]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 1 / 2950
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 7 / 2944
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 18 / 2933
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:212) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO]

In [4]:
# Format table
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


table = pd.DataFrame(res)
table["score"] = table.apply(format_score, axis=1)
table = table.pivot(index="name", columns="measure",
                    values="score").reset_index()

table[["task", "data", "model", "prompt", "topic", "extra"]] = table.apply(
    lambda x: parse_file_names(x["name"]), axis=1, result_type="expand")
table = table.drop(columns=["name", "extra"])[
    ["task", "topic", "prompt", "model", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

## Prerequisit
### Can we reproduce the results from Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [5]:
table[(table["prompt"] == "-DNA-zero-shot")
      & (table["topic"] == "topics-trec")]

measure,task,topic,prompt,model,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qrels,topics-trec,-DNA-zero-shot,gpt-4.1,0.65 ± 0.02,0.18 ± 0.01,0.82 ± 0.01
10,qrels,topics-trec,-DNA-zero-shot,gpt-oss-120b,0.70 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
28,qrels,topics-trec,-DNA-zero-shot,qwen3-14B-MT100-no-think,0.74 ± 0.02,0.14 ± 0.01,0.86 ± 0.01
37,qrels,topics-trec,-DNA-zero-shot,qwen3-30B-MT100-no-think,0.73 ± 0.02,0.11 ± 0.01,0.90 ± 0.01


### Do generated qrels benefit additional information beyond the topic title?
Binary Setting:
- `gpt-oss-120b` and `qwen3-14B-MT100-no-think` do not
- `qwen3-30B-MT100-no-think` seems to?

In [7]:
table["prompt"] = pd.Categorical(table["prompt"], ["-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative",
                                 "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"])
table[table["topic"] == "topics-trec"].sort_values(by=["model", "prompt"])

measure,task,topic,prompt,model,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qrels,topics-trec,-DNA-zero-shot,gpt-4.1,0.65 ± 0.02,0.18 ± 0.01,0.82 ± 0.01
10,qrels,topics-trec,-DNA-zero-shot,gpt-oss-120b,0.70 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
8,qrels,topics-trec,-DNA-zero-shot-masked-title,gpt-oss-120b,0.67 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
4,qrels,topics-trec,-DNA-zero-shot-masked-description,gpt-oss-120b,0.67 ± 0.02,0.16 ± 0.01,0.82 ± 0.01
5,qrels,topics-trec,-DNA-zero-shot-masked-narrative,gpt-oss-120b,0.67 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
6,qrels,topics-trec,-DNA-zero-shot-masked-title-description,gpt-oss-120b,0.61 ± 0.02,0.18 ± 0.01,0.81 ± 0.01
7,qrels,topics-trec,-DNA-zero-shot-masked-title-narrative,gpt-oss-120b,0.71 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
2,qrels,topics-trec,-DNA-zero-shot-masked-description-narrative,gpt-oss-120b,0.71 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
28,qrels,topics-trec,-DNA-zero-shot,qwen3-14B-MT100-no-think,0.74 ± 0.02,0.14 ± 0.01,0.86 ± 0.01
26,qrels,topics-trec,-DNA-zero-shot-masked-title,qwen3-14B-MT100-no-think,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.01


## Alignment
RQ: How well aligne qrels based on generated topics with qrels based on the original qrels?
- All models show a substantial drop in alignment. For example, Chohen's $\kappa$ for qwen3-30B qrels drops from the substantial agreement 0.75 to a moderate agreement of 0.52.


In [8]:
table[table["prompt"] == "-DNA-zero-shot"].sort_values(
    by=["model", "prompt", "topic"], ascending=[True, True, False])

measure,task,topic,prompt,model,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qrels,topics-trec,-DNA-zero-shot,gpt-4.1,0.65 ± 0.02,0.18 ± 0.01,0.82 ± 0.01
10,qrels,topics-trec,-DNA-zero-shot,gpt-oss-120b,0.70 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
9,qrels,,-DNA-zero-shot,gpt-oss-120b,0.37 ± 0.03,0.37 ± 0.02,0.73 ± 0.01
28,qrels,topics-trec,-DNA-zero-shot,qwen3-14B-MT100-no-think,0.74 ± 0.02,0.14 ± 0.01,0.86 ± 0.01
27,qrels,topics-robust-qwen3-14B-no-think-trec-nq5-nd3,-DNA-zero-shot,qwen3-14B-MT100-no-think,0.46 ± 0.02,0.27 ± 0.01,0.75 ± 0.01
37,qrels,topics-trec,-DNA-zero-shot,qwen3-30B-MT100-no-think,0.73 ± 0.02,0.11 ± 0.01,0.90 ± 0.01
36,qrels,topics-robust-qwen3-30B-no-think-trec-nq5-nd3,-DNA-zero-shot,qwen3-30B-MT100-no-think,0.50 ± 0.03,0.25 ± 0.01,0.78 ± 0.01
